# Task 12: High-Dimensional Latent Space Anomaly Detection via Convolutional Autoencoders

## Objective

Build a convolutional autoencoder that learns compact latent representations of images and detects anomalies using reconstruction error and latent-space distance.

The implementation includes:

- Convolutional encoder
- Structural bottleneck
- Decoder
- Reconstruction loss
- Latent-space anomaly scoring
- Pixel-wise reconstruction error
- Statistical anomaly thresholding
- ROC-AUC evaluation
- Anomaly visualization

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score, roc_curve

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

transform = transforms.ToTensor()

dataset = datasets.MNIST(
    "./data",
    train=True,
    download=True,
    transform=transform
)

dataset = Subset(dataset, range(10000))

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)


class ConvAutoencoder(nn.Module):

    def __init__(self, latent_dim=32):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),

            nn.Flatten(),
            nn.Linear(64 * 7 * 7, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64 * 7 * 7),
            nn.ReLU(),

            nn.Unflatten(
                1,
                (64, 7, 7)
            ),

            nn.ConvTranspose2d(
                64, 32, 3,
                stride=2,
                padding=1,
                output_padding=1
            ),
            nn.ReLU(),

            nn.ConvTranspose2d(
                32, 1, 3,
                stride=2,
                padding=1,
                output_padding=1
            ),
            nn.Sigmoid()
        )

    def forward(self, x):

        z = self.encoder(x)
        reconstruction = self.decoder(z)

        return reconstruction, z


model = ConvAutoencoder().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

for epoch in range(5):

    total_loss = 0

    for images, _ in loader:

        images = images.to(device)

        optimizer.zero_grad()

        reconstruction, z = model(images)

        loss = F.mse_loss(
            reconstruction,
            images
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/5 - "
        f"Loss: {total_loss / len(loader):.5f}"
    )

In [ ]:
# ============================================================
# Build normal latent distribution
# ============================================================

model.eval()

latent_vectors = []

with torch.no_grad():

    for images, _ in loader:

        images = images.to(device)

        _, z = model(images)

        latent_vectors.append(
            z.cpu().numpy()
        )

latent_vectors = np.concatenate(
    latent_vectors
)

latent_mean = latent_vectors.mean(axis=0)
latent_std = latent_vectors.std(axis=0) + 1e-6


# ============================================================
# Create normal and anomalous samples
# ============================================================

test_images = []
test_labels = []

for i in range(200):

    image, _ = dataset[i]

    test_images.append(
        image.numpy()
    )

    test_labels.append(0)


for i in range(200):

    image, _ = dataset[i]

    image = image.clone()

    # Artificial structural defect
    x = np.random.randint(5, 20)
    y = np.random.randint(5, 20)

    image[
        :,
        y:y+8,
        x:x+8
    ] = 1.0

    test_images.append(
        image.numpy()
    )

    test_labels.append(1)


test_images = torch.tensor(
    np.array(test_images),
    dtype=torch.float32
)

test_labels = np.array(
    test_labels
)


# ============================================================
# Latent + reconstruction anomaly scores
# ============================================================

with torch.no_grad():

    reconstructions, z = model(
        test_images.to(device)
    )

    z = z.cpu().numpy()

    reconstructions = (
        reconstructions
        .cpu()
        .numpy()
    )

    originals = test_images.numpy()

latent_scores = np.sqrt(
    np.mean(
        (
            (z - latent_mean)
            / latent_std
        ) ** 2,
        axis=1
    )
)

reconstruction_scores = np.mean(
    (
        originals
        - reconstructions
    ) ** 2,
    axis=(1, 2, 3)
)

combined_scores = (
    latent_scores
    + reconstruction_scores * 10
)

auc = roc_auc_score(
    test_labels,
    combined_scores
)

fpr, tpr, _ = roc_curve(
    test_labels,
    combined_scores
)

print("ROC-AUC:", round(auc, 4))

In [ ]:
# ============================================================
# Threshold and visualization
# ============================================================

threshold = np.percentile(
    combined_scores[:200],
    95
)

predictions = (
    combined_scores > threshold
)

print(
    "Anomaly threshold:",
    round(threshold, 4)
)

print(
    "Detected anomalies:",
    predictions.sum()
)


plt.figure(figsize=(6, 5))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {auc:.3f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    "--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Anomaly Detection ROC Curve")
plt.legend()
plt.grid(True)
plt.show()


# Visualize examples
plt.figure(figsize=(10, 4))

for i in range(5):

    plt.subplot(2, 5, i + 1)

    plt.imshow(
        originals[i, 0],
        cmap="gray"
    )

    plt.title("Normal")
    plt.axis("off")

    plt.subplot(2, 5, i + 6)

    plt.imshow(
        originals[200 + i, 0],
        cmap="gray"
    )

    plt.title("Anomaly")
    plt.axis("off")

plt.tight_layout()
plt.show()

# Conclusion

A convolutional autoencoder was implemented for high-dimensional anomaly detection.

The encoder learned a compact latent representation, while the decoder reconstructed the original image. Anomalies were identified using a combination of latent-space deviation and reconstruction error.

ROC-AUC was used to quantitatively evaluate anomaly separation, while visual inspection demonstrated the ability to identify structural defects.